# Blinded CPIC pharmacogenomics study — 110 cases

Reproduces `claude_science_answers.csv` and the three lookup tables.

Pipeline: `genotype → diplotype` by constraint solving over the CPIC allele-definition table; `diplotype → phenotype` and `phenotype → recommendation` by lookup against the CPIC data API.

See `METHODS.md` for assumptions, sources and the error self-assessment.

In [1]:
import json
import pandas as pd
import pgx_pipeline as P
import run_study as R

cases = json.load(open('cases_blinded.json'))
print(len(cases), 'cases;', pd.Series([c['gene'] for c in cases]).nunique(), 'genes;',
      pd.Series([c['drug'] for c in cases]).nunique(), 'drugs')

110 cases; 21 genes; 32 drugs


## 1. CPIC reference data

Cached in `cpic_cache.json`; delete that file to re-download.

In [2]:
data, funcs, models, diplo = R.load_reference()
print('CPIC API retrieved:', data['retrieved'])
print({k: len(v) for k, v in data.items() if isinstance(v, list)})
print('diplotype rows per gene:', {g: len(v) for g, v in diplo.items()})

CPIC API retrieved: 2026-09-15T17:15:02.528671+00:00
{'allele_definition': 1290, 'allele': 1314, 'drug': 324, 'pair': 635, 'recommendation': 571}
diplotype rows per gene: {'CYP2D6': 16836, 'CYP2C19': 666, 'CYP2C9': 2850, 'CYP2B6': 1176, 'CYP3A5': 21, 'CYP4F2': 0, 'DPYD': 3570, 'HLA-A': 2, 'HLA-B': 6, 'IFNL3': 0, 'MT-RNR1': 24, 'NUDT15': 276, 'SLCO1B1': 741, 'TPMT': 1225, 'UGT1A1': 45, 'VKORC1': 0}


## 2. The allele model

CPIC allele definitions are rsID-indexed. IUPAC ambiguity codes mark positions where sub-alleles disagree, so a position whose code still permits the reference base is not a defining variant. CYP2D6\*4 is `T` at rs3892097 but ambiguous `R` at rs1065852.

In [3]:
m = models['CYP2D6']
print('*4  rs3892097:', m.value('*4', 'rs3892097'), '->', m.expand('*4', 'rs3892097'))
print('*4  rs1065852:', m.value('*4', 'rs1065852'), '->', m.expand('*4', 'rs1065852'))
print('*4  definite defining loci:', sorted(m.defining_loci('*4')))
print('*10 definite defining loci:', sorted(m.defining_loci('*10')))

*4  rs3892097: T -> {'T'}
*4  rs1065852: R -> {'A', 'G'}
*4  definite defining loci: ['rs3892097']
*10 definite defining loci: ['rs1065852', 'rs1135840']


## 3. Solving a diplotype

Ranking is (unconfirmed defining variants, non-reference alleles, total defining variants). The third example is the compound CYP2D6 case, the one place two clinically distinct diplotypes are compatible with the observation; they share a phenotype.

In [4]:
for gt in [{'rs3892097': 'T/T', 'rs1065852': 'G/G'},
           {'rs3892097': 'C/T', 'rs1065852': 'G/G'},
           {'rs3892097': 'C/T', 'rs1065852': 'A/G'}]:
    sols = m.solve(gt)
    print(gt, '->', [(s['call'], s['unconfirmed'], s['n_variant_alleles']) for s in sols[:3]])

{'rs3892097': 'T/T', 'rs1065852': 'G/G'} -> [('*4/*4', 0, 2)]
{'rs3892097': 'C/T', 'rs1065852': 'G/G'} -> [('*1/*4', 0, 1), ('*106/*4', 1, 2), ('*107/*4', 1, 2)]
{'rs3892097': 'C/T', 'rs1065852': 'A/G'} -> [('*1/*4', 0, 1), ('*106/*4', 1, 2), ('*107/*4', 1, 2)]


## 4. Run the study

In [5]:
answers, annotated, data, diplo, recs = R.main()
answers.head(8)

                       case_id  ... abstained
0            cyp2d6_codeine_pm  ...     false
1            cyp2d6_codeine_im  ...     false
2            cyp2d6_codeine_nm  ...     false
3            cyp2d6_codeine_um  ...     false
4  cyp2d6_codeine_im_decreased  ...     false
5              cyp2d6_tamox_pm  ...     false
6              cyp2d6_tamox_im  ...     false
7              cyp2d6_tamox_nm  ...     false

[8 rows x 5 columns]

## 5. Summary

In [6]:
u = pd.to_numeric(annotated.unconfirmed_defining_variants, errors='coerce').fillna(0)
print('cases                :', len(answers))
print('abstentions          :', (answers.abstained == 'true').sum())
print('calling method       :', annotated.calling_method.value_counts().to_dict())
print('unique diplotypes    :', annotated.diplotype.nunique())
print('CPIC pair level      :', annotated.cpic_level.value_counts().to_dict())
print('lethal-class cases   :', (annotated.lethal_class == 'true').sum())
print('unconfirmed >0       :', int((u > 0).sum()))
print('equal-rank alt calls :', int((annotated.alternative_diplotypes.fillna('') != '').sum()))

cases                : 110
abstentions          : 0
calling method       : {'solver': 83, 'rule': 27}
unique diplotypes    : 41
CPIC pair level      : {'A': 108, 'Retired': 2}
lethal-class cases   : 24
unconfirmed >0       : 11
equal-rank alt calls : 0


In [7]:
annotated.groupby('phenotype').size().sort_values(ascending=False)

phenotype
Intermediate Metabolizer                                 22
Poor Metabolizer                                         20
Normal Metabolizer                                       20
Ultrarapid Metabolizer                                    9
Rapid Metabolizer                                         5
Deficient                                                 3
*15:02 negative                                           2
Poor Function                                             2
*15:02 positive                                           2
Normal                                                    2
Normal Function                                           2
Decreased Function                                        2
Favourable response genotype                              1
*31:01 negative                                           1
ivacaftor responsive in CF patients                       1
ivacaftor non-responsive in CF patients                   1
increased risk of aminoglycosi

### Lethal-class cases

Standard dosing carries a documented risk of death or of irreversible major organ or sensory injury.

In [8]:
L = annotated[annotated.lethal_class == 'true']
print(L[['case_id', 'gene', 'drug', 'phenotype']].to_string(index=False))

             case_id        gene                  drug                                             phenotype
   cyp2d6_codeine_um      CYP2D6               codeine                                Ultrarapid Metabolizer
     cyp2c19_clop_pm     CYP2C19           clopidogrel                                      Poor Metabolizer
     cyp2c19_clop_im     CYP2C19           clopidogrel                              Intermediate Metabolizer
      cyp2c9_warf_pm      CYP2C9              warfarin                                      Poor Metabolizer
      cyp2c9_phen_pm      CYP2C9             phenytoin                                      Poor Metabolizer
          dpyd_fu_pm        DPYD          fluorouracil                                      Poor Metabolizer
       dpyd_fu_im_2a        DPYD          fluorouracil                              Intermediate Metabolizer
    dpyd_fu_im_hapb3        DPYD          fluorouracil                              Intermediate Metabolizer
        dpyd_cape_p

### Calls resting on an unobserved defining variant

In [9]:
print(annotated.loc[u > 0, ['case_id', 'diplotype', 'phenotype',
                            'unconfirmed_defining_variants']].to_string(index=False))

         case_id                                   diplotype                phenotype unconfirmed_defining_variants
 cyp2c19_clop_rm                                      *1/*17        Rapid Metabolizer                             1
 cyp2c19_clop_um                                     *17/*17   Ultrarapid Metabolizer                             2
 cyp2c19_vori_rm                                      *1/*17        Rapid Metabolizer                             1
 cyp2c19_vori_um                                     *17/*17   Ultrarapid Metabolizer                             2
cyp2c19_cital_rm                                      *1/*17        Rapid Metabolizer                             1
cyp2c19_cital_um                                     *17/*17   Ultrarapid Metabolizer                             2
 cyp2c19_omep_rm                                      *1/*17        Rapid Metabolizer                             1
 cyp2c19_omep_um                                     *17/*17   Ultrarapi

## 6. Output tables

In [10]:
for f in ['claude_science_answers.csv', 'claude_science_answers_annotated.csv',
          'table_allele_definitions.csv', 'table_diplotype_to_phenotype.csv',
          'table_phenotype_to_recommendation.csv']:
    print(f, pd.read_csv(f).shape)

claude_science_answers.csv (110, 5)
claude_science_answers_annotated.csv (110, 22)
table_allele_definitions.csv (34, 6)
table_diplotype_to_phenotype.csv (74, 7)
table_phenotype_to_recommendation.csv (552, 6)


In [11]:
pd.read_csv('table_diplotype_to_phenotype.csv').head(15)

       gene  ...       alternative_diplotypes
0    CYP2D6  ...  *10/*100; *10/*101; *10/*56
1    CYP2D6  ...     *1/*100; *1/*101; *1/*56
2    CYP2D6  ...    *1/*106; *1/*107; *1/*110
3    CYP2D6  ...     *4/*100; *4/*101; *4/*56
4    CYP2D6  ...    *4/*106; *4/*107; *4/*110
5    CYP2D6  ...    *4/*106; *4/*107; *4/*110
6    CYP2D6  ...                          NaN
7    CYP2D6  ...                          NaN
8    CYP2D6  ...                          NaN
9   CYP2C19  ...                          NaN
10  CYP2C19  ...        *1/*2; *2/*16; *2/*30
11  CYP2C19  ...        *2/*4; *2/*44; *2/*45
12  CYP2C19  ...        *1/*1; *1/*16; *1/*30
13  CYP2C19  ...        *1/*4; *1/*44; *1/*45
14  CYP2C19  ...     *17/*44; *17/*45; *4/*17

[15 rows x 7 columns]